<a href="https://colab.research.google.com/github/Marti6cp/BANA7075_ProjectTeam6/blob/main/BANA7075_FinalProject_Team6_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
# !pip install great_expectations
# !pip install dvc dvc-gdrive

In [56]:
#####GitHub Configuration to Maintain Versioning --Martinez

# token = "REDACTED_TOKEN"
# !git config --global user.email "marti6cp@mail.uc.edu"
# !git config --global user.name "Chris Martinez"

# username = "marti6cp"
# repo = "BANA7075_ProjectTeam6"

# !git clone https://{token}@github.com/{username}/{repo}.git

# %cd /content/BANA7075_ProjectTeam6




In [57]:
####DVC initialization ---Martinez

# !dvc init
# !git add .dvc .dvcignore
# !git commit -m "Initialize DVC"

In [58]:
#####KaggleHub DataIngest Module- Martinez

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load a DataFrame with a specific version of a CSV
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "sharmajicoder/amazon-e-commerce", "amazon_ecommerce_1M.csv",
)
#print(df.head())


####DataProcessing
### Partition training and testing data random state maintains sample reproducability
df_training = df.sample(frac=0.8, random_state=25)##
df_testing = df.drop(df_training.index)

df_training.to_csv('train.csv', index=False)
df_testing.to_csv('test.csv', index=False)


## Drop Null frames replacing in existing data frame
df_training.dropna(inplace=True)
df_testing.dropna(inplace=True)

#drop duplicates---not sure on implementation yet. purchase date does not contain
#enough granularity to avoid removing real data especially in a large ecommerce dataset

# df_training.drop_duplicates(inplace=True)
# df_testing.drop_duplicates(inplace=True)

##data type consistency
##Text/string
for col in ["user_id", "product_id", "category", "subcategory", "brand",
            "seller_id", "location", "device" ,"payment_method", "delivery_status"]:
    df_training[col] = df_training[col].astype("string")


##numerics
for col in ["price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_rating","shipping_time_days",
                  ]:
    df_training[col] = pd.to_numeric(df_training[col], errors='coerce')

## Dates
df_training["purchase_date"] = pd.to_datetime(df_training["purchase_date"])

print(df_training.dtypes)



Using Colab cache for faster access to the 'amazon-e-commerce' dataset.
user_id               string[python]
product_id            string[python]
category              string[python]
subcategory           string[python]
brand                 string[python]
price                        float64
discount                     float64
final_price                  float64
rating                       float64
review_count                   int64
stock                          int64
seller_id             string[python]
seller_rating                float64
purchase_date         datetime64[ns]
shipping_time_days             int64
location              string[python]
device                string[python]
payment_method        string[python]
is_returned                     bool
delivery_status       string[python]
dtype: object


In [59]:
df.to_csv("/content/BANA7075_ProjectTeam6/dataset.csv", index=False)
df_training.to_csv("/content/BANA7075_ProjectTeam6/train.csv", index=False)
df_testing.to_csv("/content/BANA7075_ProjectTeam6/test.csv", index=False)

# 3. Tell DVC to track the newly saved file
!dvc add /content/BANA7075_ProjectTeam6/dataset.csv
!dvc add /content/BANA7075_ProjectTeam6/train.csv
!dvc add /content/BANA7075_ProjectTeam6/test.csv

# !dvc remote add -d mygdriveremote gdrive://1wDwC_wUNndVpWGCrxqWkl2ufNSLgzWgB
# 4. Commit the DVC pointer metadata to Git
!git add /content/BANA7075_ProjectTeam6/dataset.csv.dvc #/content/BANA7075_ProjectTeam6/.gitignore
!git add /content/BANA7075_ProjectTeam6/train.csv.dvc #/content/BANA7075_ProjectTeam6/.gitignore
!git add /content/BANA7075_ProjectTeam6/test.csv.dvc #/content/BANA7075_ProjectTeam6/.gitignore

!git commit -am "Updated Pandas raw DataFrame,Train & test via DVC"

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
100% 1/1 [00:00<00:00,  1.14files/s{'info': ''}]
Adding...: 100% 1/1 [00:01<00:00,  1.69s/file{'info': ''}]

To track the changes with git, run:

	git add dataset.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
100% 1/1 [00:00<00:00,  1.18files/s{'info': ''}]
Adding...: 100% 1/1 [00:01<00:00,  1.60s/file{'info': ''}]

To track the changes with git, run:

	git add tra

In [60]:
##### DVC push DVC google drive access blocked workaround
!dvc push
!git push origin main

from google.colab import files
files.download("dataset.csv")
files.download("train.csv")
files.download("test.csv")

Pushing
ERROR: failed to push data to the cloud - config file error: no remote specified in /content/BANA7075_ProjectTeam6. Create a default remote with
    dvc remote add -d <remote name> <remote url>
Everything up-to-date


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [61]:
#from great_expectations.expectations.row_conditions import Column
#######DataValidation Module######## - Martinez

import great_expectations as gx

###Setup GX Environment
context = gx.get_context()

###Connect to Data
# Variables
datasource_name = "ecommerce_datasource"
data_asset_name = "ecommerce_df_asset"
batch_definition_name = "big_batch"
#creating datasource
datasource = context.data_sources.add_pandas(
    name= datasource_name
    )
data_asset = datasource.add_dataframe_asset(
    name= data_asset_name
    )

#Create Batch Definition/Batch
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df_training}
                                   )


#### Define Expectations

##VRule to validate All training records are present
RecordMatch = gx.expectations.ExpectTableRowCountToEqual(
    value = 800000

)

##Rule to validate all Data Columnn are present and in order
ColumnMatch = gx.expectations.ExpectTableColumnsToMatchOrderedList(
    column_list=["user_id", "product_id", "category", "subcategory", "brand",
                 "price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_id", "seller_rating","purchase_date","shipping_time_days",
                 "location", "device" ,"payment_method","is_returned",
                 "delivery_status"  ]
)

##Rule to validate that trainging dataframe does not contain missing data
column_list = df.columns.tolist()
for columns in column_list:       #iterates through the columns in data to check for missing data
    NoNulls = gx.expectations.ExpectColumnValuesToNotBeNull(
    column=columns
)


## Rule to validate Data Set still has correct nuber of columns
Column_number_check = gx.expectations.ExpectTableColumnCountToEqual(
    value=20
)

## Rule to validate column as strings dType
for col in ["user_id", "product_id", "category", "subcategory", "brand",
            "seller_id", "location", "device" ,"payment_method", "delivery_status"]:
    string_check = gx.expectations.ExpectColumnValuesToBeOfType(
    column= col,
    type_= "str"
    )


##. Rule to validate column values are numeric dType
for col in ["price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_rating","shipping_time_days",]:
    num_check = gx.expectations.ExpectColumnValuesToBeInTypeList(
    column= col,
    type_list = ["float", "int"]
    )




## Rule to validate column only contains dates dType
date_check = gx.expectations.ExpectColumnValuesToBeOfType(
    column= "purchase_date",
    type_= "datetime64"
    )
# validation_results = batch.validate(date_check)
# print(validation_results)


##Rule to validate "is_returned is bool data type
isBooleanDtype = gx.expectations.ExpectColumnValuesToBeOfType(
    column= "is_returned",
    type_= "bool"
)


# #### Create Expectation Suite --testing
# suite_name = "data_valid_expectation_suite"
# suite = gx.ExpectationSuite(
#     name=suite_name
#     )
# suite = context.suites.add(
#     suite
#     )
# suite.add_expectation

# ####
# definition_name = "my_validation_definition"
# validation_definition = gx.ValidationDefinition(
#     data=batch, suite=suite_name, name=definition_name
# )

# batchValidResults = validation_definition.run(batch_parameters=batch_parameters_yearly)
# print(batchValidResults)
##Run Validations
validation_results = batch.validate(string_check)
print(validation_results)
if validation_results.success:
  print("Data passed validation!")
else:

  print("Data Validation Failed!!!")
# print(ColumnMatch)


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpqs48i1dw' for ephemeral docs site


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_of_type",
    "kwargs": {
      "batch_id": "ecommerce_datasource-ecommerce_df_asset",
      "column": "delivery_status",
      "type_": "str"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": "str"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}
Data passed validation!
